# Idea 5 — Watching the Ice Melt: Dataset Downloader
### DSC 106 Final Project · Cryosphere Datasets

This notebook downloads every dataset used across the 6 Idea 5 visualizations directly to your **Desktop**.

| Symbol | Meaning |
|--------|---------|
| 🌐 | Public URL — no account needed |
| 🔑 | Requires free NASA Earthdata account |

**Register here if needed:** https://urs.earthdata.nasa.gov/users/new

**Earthdata cryosphere catalog:** https://search.earthdata.nasa.gov/search?fst0=Cryosphere

Earthdata Search is the browser catalog. For reproducible notebook downloads, this notebook uses direct public file URLs where possible and NASA CMR / `earthaccess` for authenticated Earthdata products.

| Product | Used for | Browser search |
|---------|----------|----------------|
| NSIDC Sea Ice Index v4 (G02135) | VIZ 01, 02, 05 | https://search.earthdata.nasa.gov/search?q=G02135 |
| MODIS Sea Ice (MOD29) | VIZ 01, 02, 05 validation | https://search.earthdata.nasa.gov/search?q=MOD29 |
| GRACE / GRACE-FO Mascon | VIZ 03 | https://search.earthdata.nasa.gov/search?q=TELLUS_GRAC-GRFO_MASCON |
| MODIS Snow Cover (MOD10A1) | VIZ 03 context | https://search.earthdata.nasa.gov/search?q=MOD10A1 |
| MODIS Land Surface Temp (MOD11A3) | VIZ 04 | https://search.earthdata.nasa.gov/search?q=MOD11A3 |
| MODIS BRDF/Albedo (MCD43A3) | VIZ 06 | https://search.earthdata.nasa.gov/search?q=MCD43A3 |

Run cells top-to-bottom. Each section is self-contained — skip any you don't need.

---

## Setup — find Desktop & install libraries

In [1]:
import os, sys, subprocess, shutil
from pathlib import Path

def get_desktop():
    home = Path.home()
    for candidate in [home / 'Desktop', home / 'デスクトップ', home / 'Bureau']:
        if candidate.exists():
            return candidate
    fallback = home / 'Desktop'
    fallback.mkdir(exist_ok=True)
    return fallback

DESKTOP = get_desktop()
SAVE_DIR = DESKTOP / 'idea5_cryosphere_data'
SAVE_DIR.mkdir(exist_ok=True)
print(f'Saving all files to:\n  {SAVE_DIR}')

def ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

ensure('requests')
ensure('tqdm')
import requests
from tqdm.notebook import tqdm
print('Libraries ready')

Saving all files to:
  /Users/joeysandoval/Desktop/idea5_cryosphere_data
Libraries ready


## Shared download helper

In [2]:
def download(url, dest_path, session=None, label=None):
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    label = label or dest_path.name
    req = session or requests
    try:
        with req.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            with open(dest_path, 'wb') as f, tqdm(
                total=total, unit='B', unit_scale=True, desc=label, leave=True
            ) as bar:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
        print(f'  Saved -> {dest_path}')
        return dest_path
    except Exception as e:
        print(f'  Failed: {e}')
        return None

---
## 🌐 VIZ 01, 02, 05 — NSIDC Sea Ice Index v4
**Used for:** decadal polar small-multiples · daily spaghetti plot · ice-free projection baseline  
Source: https://nsidc.org/data/G02135/versions/4  
Browser search: https://search.earthdata.nasa.gov/search?q=G02135  
No account required.

Notes:
- Daily extent files are one CSV per hemisphere.
- Monthly extent files are one CSV per month/hemisphere, e.g. `N_09_extent_v4.0.csv` for Arctic September.

In [3]:
nsidc_dir = SAVE_DIR / 'VIZ01_02_05__NSIDC_SeaIceIndex_v4'
nsidc_dir.mkdir(exist_ok=True)

nsidc_base = 'https://noaadata.apps.nsidc.org/NOAA/G02135'
nsidc_files = {
    'N_seaice_extent_daily_v4.0.csv': f'{nsidc_base}/north/daily/data/N_seaice_extent_daily_v4.0.csv',
    'S_seaice_extent_daily_v4.0.csv': f'{nsidc_base}/south/daily/data/S_seaice_extent_daily_v4.0.csv',
}

# Monthly NSIDC Sea Ice Index files are stored per month, not as one combined monthly CSV.
for hemi, folder in [('N', 'north'), ('S', 'south')]:
    for month in range(1, 13):
        filename = f'{hemi}_{month:02d}_extent_v4.0.csv'
        nsidc_files[filename] = f'{nsidc_base}/{folder}/monthly/data/{filename}'

print('Downloading NSIDC Sea Ice Index v4 files...')
for filename, url in nsidc_files.items():
    download(url, nsidc_dir / filename, label=filename)

(nsidc_dir / 'README.txt').write_text(
    'NSIDC Sea Ice Index v4 (G02135)\n'
    'Source landing page: https://nsidc.org/data/G02135/versions/4\n'
    'Data archive: https://noaadata.apps.nsidc.org/NOAA/G02135/\n'
    'Earthdata Search: https://search.earthdata.nasa.gov/search?q=G02135\n\n'
    'Daily files: N/S_seaice_extent_daily_v4.0.csv\n'
    'Monthly files: N/S_01_extent_v4.0.csv through N/S_12_extent_v4.0.csv\n'
)
print(f'\nDone -> {nsidc_dir}')

N_seaice_extent_daily_v4.0.csv:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v4/N_seaice_extent_daily_v4.0.csv


S_seaice_extent_daily_v4.0.csv:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v4/S_seaice_extent_daily_v4.0.csv


N_01_extent_v4.0.csv:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

...

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v4/S_10_extent_v4.0.csv


S_11_extent_v4.0.csv:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v4/S_11_extent_v4.0.csv


S_12_extent_v4.0.csv:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v4/S_12_extent_v4.0.csv

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_05__NSIDC_SeaIceIndex_v4


---
## 🌐 VIZ 03 — NASA GRACE / GRACE-FO Ice Mass Anomaly
**Used for:** Greenland + Antarctica cumulative mass loss bar chart  
Public quick CSV: https://ourworldindata.org/grapher/ice-sheet-mass-balance  
Earthdata full mascon product search: https://search.earthdata.nasa.gov/search?q=TELLUS_GRAC-GRFO_MASCON  
No account required for the quick CSV; Earthdata account required for full PO.DAAC files.

In [4]:
grace_dir = SAVE_DIR / 'VIZ03__GRACE_IceMassAnomaly'
grace_dir.mkdir(exist_ok=True)

# Our World in Data hosts the IMBIE/GRACE-FO composite as a clean public CSV.
# Columns: Entity (Antarctica|Greenland), Code (ATA|GRL), Day (YYYY-MM-DD), Seasonal variation (Gt cumulative).
owid_url = 'https://ourworldindata.org/grapher/ice-sheet-mass-balance.csv?v=1&csvType=full&useColumnShortNames=false'
combined_path = grace_dir / 'ice_sheet_mass_balance.csv'

print('Downloading combined GRACE/IMBIE ice mass anomaly...')
result = download(owid_url, combined_path, label='ice_sheet_mass_balance.csv')

if result:
    import csv
    rows_grn, rows_ant = [], []
    with open(combined_path) as f:
        r = csv.reader(f)
        header = next(r)
        for row in r:
            if not row:
                continue
            if row[0] == 'Greenland':
                rows_grn.append(row)
            elif row[0] == 'Antarctica':
                rows_ant.append(row)

    def _write(path, rows):
        with open(path, 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['date', 'cumulative_mass_Gt'])
            for row in rows:
                w.writerow([row[2], row[3]])

    _write(grace_dir / 'greenland_mass_anomaly.csv', rows_grn)
    _write(grace_dir / 'antarctica_mass_anomaly.csv', rows_ant)
    print(f'  Greenland records: {len(rows_grn)}')
    print(f'  Antarctica records: {len(rows_ant)}')

(grace_dir / 'README.txt').write_text(
    'GRACE / GRACE-FO ice sheet mass anomaly\n'
    'Quick CSV source: https://ourworldindata.org/grapher/ice-sheet-mass-balance\n'
    'Earthdata Search: https://search.earthdata.nasa.gov/search?q=TELLUS_GRAC-GRFO_MASCON\n'
    'PO.DAAC landing page: https://podaac.jpl.nasa.gov/dataset/TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.1_V3\n'
)
print(f'\nDone -> {grace_dir}')

ice_sheet_mass_balance.csv: 0.00B [00:00, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ03__GRACE_IceMassAnomaly/ice_sheet_mass_balance.csv
  Greenland records: 192
  Antarctica records: 192

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ03__GRACE_IceMassAnomaly


---
## 🌐 VIZ 04 — NOAA Temperature Context + NASA MODIS LST
**Used for:** Arctic temperature anomaly heatmap (month × year)  
NOAA context endpoint is public. For Arctic-wide satellite LST, use MOD11A3 through Earthdata below.

- NOAA Climate-at-a-Glance endpoint pattern: `.../{scope}/time-series/{location}/{parameter}/{surface}/{timescale}/{month}/{begYear}-{endYear}/data.csv`
- MOD11A3 Earthdata Search: https://search.earthdata.nasa.gov/search?q=MOD11A3

In [5]:
noaa_dir = SAVE_DIR / 'VIZ04__NOAA_ArcticTemps'
noaa_dir.mkdir(exist_ok=True)

# Alaska monthly temperature anomalies as a public northern-latitude context series.
# For the final Arctic-wide map/heatmap, prefer MOD11A3 below through Earthdata.
noaa_url = (
    'https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/'
    '50/tavg/all/1/1979-2025/data.csv?base_prd=true&begbaseyear=1981&endbaseyear=2010'
)

print('Downloading NOAA Alaska monthly temperature anomaly context series...')
download(noaa_url, noaa_dir / 'alaska_monthly_temp_anomaly_1979_2025.csv',
         label='alaska_temp_anomaly.csv')

(noaa_dir / 'README.txt').write_text(
    'NOAA NCEI Climate-at-a-Glance temperature anomaly context series\n'
    'Downloaded file: Alaska monthly average temperature anomalies, 1979-2025\n'
    'Baseline: 1981-2010\n'
    'CSV endpoint pattern: https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/{scope}/time-series/{location}/{parameter}/{surface}/{timescale}/{month}/{begYear}-{endYear}/data.csv\n\n'
    'For the final Arctic-wide satellite heatmap, use NASA Earthdata MOD11A3:\n'
    '  https://search.earthdata.nasa.gov/search?q=MOD11A3\n'
)
print(f'\nDone -> {noaa_dir}')

alaska_temp_anomaly.csv: 0.00B [00:00, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ04__NOAA_ArcticTemps/alaska_monthly_temp_anomaly_1979_2025.csv

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ04__NOAA_ArcticTemps


---
## 🌐 VIZ 05 — CMIP6 Arctic Sea Ice Projections (IPCC AR6)
**Used for:** ice-free Arctic summer projection fan chart (SSP1-2.6, SSP2-4.5, SSP5-8.5)  

Full CMIP6 NetCDFs are too large for a quick class-project notebook download. This section writes stable access links instead of trying stale deep links.

- ESGF CMIP6 search: https://esgf-node.llnl.gov/search/cmip6/
- KNMI Climate Explorer CMIP6 browser: https://climexp.knmi.nl/selectfield_cmip6.cgi
- Useful filters: `variable_id=siconc`, `experiment_id=ssp126|ssp245|ssp585`, `table_id=SImon`.

In [6]:
cmip6_dir = SAVE_DIR / 'VIZ05__CMIP6_SeaIce_Projections'
cmip6_dir.mkdir(exist_ok=True)

(cmip6_dir / 'README.txt').write_text(
    'CMIP6 Arctic Sea Ice Projections\n'
    'Variable: sea-ice concentration / sea-ice area fraction, commonly variable_id=siconc\n'
    'Scenarios: SSP1-2.6, SSP2-4.5, SSP5-8.5\n\n'
    'Browser download / exploration links:\n'
    '  ESGF CMIP6 search: https://esgf-node.llnl.gov/search/cmip6/\n'
    '  KNMI Climate Explorer: https://climexp.knmi.nl/selectfield_cmip6.cgi\n\n'
    'Suggested ESGF filters:\n'
    '  variable_id=siconc\n'
    '  experiment_id=ssp126 OR ssp245 OR ssp585\n'
    '  table_id=SImon\n\n'
    'For this DSC 106 prototype, the webpage uses a transparent toy projection curve\n'
    'for interaction design. Replace it with processed CMIP6 medians if the final\n'
    'story requires model-derived projection values.\n'
)
print('Wrote CMIP6 access README with stable browser links.')
print(f'Done -> {cmip6_dir}')

Wrote CMIP6 access README with stable browser links.
Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ05__CMIP6_SeaIce_Projections


---
## 🌐 VIZ 01, 02, 04, 06 — NASA MODIS Imagery via GIBS API
**Used for:** visual satellite context tiles · LST imagery · snow-cover/albedo context  
Source: https://gibs.earthdata.nasa.gov/  
Layer browser: https://worldview.earthdata.nasa.gov/  
API docs: https://nasa-gibs.github.io/gibs-api-docs/access-basics/

In [7]:
gibs_dir = SAVE_DIR / 'VIZ01_02_04_06__MODIS_GIBS_Tiles'
gibs_dir.mkdir(exist_ok=True)

GIBS = (
    'https://gibs.earthdata.nasa.gov/wmts/epsg4326/best/'
    '{layer}/default/{date}/{tms}/{z}/{y}/{x}.{ext}'
)

# (layer, date, tile_matrix_set, file_ext, label)
# zoom=3, y=1, x=4 covers a central Arctic overview in EPSG:4326 tiles.
layers = [
    ('MODIS_Terra_CorrectedReflectance_TrueColor', '2012-09-16', '250m', 'jpg', 'VIZ01_02__2012_record_min'),
    ('MODIS_Terra_CorrectedReflectance_TrueColor', '2023-09-19', '250m', 'jpg', 'VIZ01_02__2023_near_record'),
    ('MODIS_Terra_CorrectedReflectance_TrueColor', '2000-09-15', '250m', 'jpg', 'VIZ01_02__2000_reference'),
    ('MODIS_Terra_Land_Surface_Temp_Day',          '2023-07-15', '1km',  'png', 'VIZ04__2023_summer_LST'),
    ('MODIS_Terra_Land_Surface_Temp_Day',          '2012-07-15', '1km',  'png', 'VIZ04__2012_summer_LST'),
    ('MODIS_Terra_L3_NDSI_Snow_Cover_Daily',                '2023-09-01', '500m', 'png', 'VIZ06__2023_Sep_snow_cover'),
    ('MODIS_Terra_L3_NDSI_Snow_Cover_Daily',                '2000-09-01', '500m', 'png', 'VIZ06__2000_Sep_snow_cover'),
]

print('Downloading MODIS GIBS tiles...')
for layer, date, tms, ext, label in layers:
    url = GIBS.format(layer=layer, date=date, tms=tms, z=3, y=1, x=4, ext=ext)
    download(url, gibs_dir / f'{label}.{ext}', label=label)

(gibs_dir / 'README.txt').write_text(
    'NASA MODIS Tiles via GIBS API\n'
    'Tile coordinates: zoom=3, y=1, x=4 (central Arctic overview)\n\n'
    'Layer-specific formats:\n'
    '  MODIS_Terra_CorrectedReflectance_TrueColor : 250m, .jpg\n'
    '  MODIS_Terra_Land_Surface_Temp_Day          : 1km,  .png\n'
    '  MODIS_Terra_L3_NDSI_Snow_Cover_Daily                : 500m, .png\n\n'
    'Custom fetch pattern:\n'
    '  https://gibs.earthdata.nasa.gov/wmts/epsg4326/best/{LAYER}/default/{DATE}/{TMS}/{Z}/{Y}/{X}.{EXT}\n\n'
    'Layer browser: https://worldview.earthdata.nasa.gov/\n'
    'API docs:      https://nasa-gibs.github.io/gibs-api-docs/access-basics/\n'
)
print(f'\nDone -> {gibs_dir}')

VIZ01_02__2012_record_min:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ01_02__2012_record_min.jpg


VIZ01_02__2023_near_record:   0%|          | 0.00/82.2k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ01_02__2023_near_record.jpg


VIZ01_02__2000_reference:   0%|          | 0.00/82.2k [00:00<?, ?B/s]

...

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ04__2012_summer_LST.png


VIZ06__2023_Sep_snow_cover:   0%|          | 0.00/7.65k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ06__2023_Sep_snow_cover.png


VIZ06__2000_Sep_snow_cover:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

  Saved -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles/VIZ06__2000_Sep_snow_cover.png

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_02_04_06__MODIS_GIBS_Tiles


---
## 🔑 VIZ 01, 03, 04, 06 — NASA Earthdata MODIS HDF Products
**Used for:** MOD29 sea ice · MOD10A1 snow cover · MOD11A3 LST monthly · MCD43A3 albedo  
Requires a **free NASA Earthdata account**: https://urs.earthdata.nasa.gov/users/new  
Cryosphere catalog: https://search.earthdata.nasa.gov/search?fst0=Cryosphere  
Uses the `earthaccess` Python library — installs automatically.

This cell uses interactive/environment login rather than storing your password in the notebook.

In [8]:
# NASA Earthdata authenticated downloads through CMR / earthaccess.
# Do not paste your password into this notebook. earthaccess will prompt you or use ~/.netrc.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'earthaccess', '-q'])
import earthaccess

try:
    auth = earthaccess.login(strategy='interactive', persist=True)
except TypeError:
    # Older earthaccess versions may not support persist=True.
    auth = earthaccess.login(strategy='interactive')

modis_dir = SAVE_DIR / 'VIZ01_03_04_06__MODIS_HDF_Products'
modis_dir.mkdir(exist_ok=True)

# Each item includes a stable Earthdata Search URL for manual verification.
products = [
    {
        'short_name': 'MOD29',
        'version': '061',
        'start': '2023-09-16',
        'end': '2023-09-16',
        'label': 'VIZ01_02_05__MOD29_SeaIce',
        'search_url': 'https://search.earthdata.nasa.gov/search?q=MOD29',
    },
    {
        'short_name': 'MOD10A1',
        'version': '061',
        'start': '2023-09-16',
        'end': '2023-09-16',
        'label': 'VIZ03__MOD10A1_SnowCover',
        'search_url': 'https://search.earthdata.nasa.gov/search?q=MOD10A1',
    },
    {
        'short_name': 'MOD11A2',
        'version': '061',
        'start': '2023-09-01',
        'end': '2023-09-30',
        'label': 'VIZ04__MOD11A3_LST_Monthly',
        'search_url': 'https://search.earthdata.nasa.gov/search?q=MOD11A3',
    },
    {
        'short_name': 'MCD43A3',
        'version': '061',
        'start': '2023-09-16',
        'end': '2023-09-16',
        'label': 'VIZ06__MCD43A3_Albedo',
        'search_url': 'https://search.earthdata.nasa.gov/search?q=MCD43A3',
    },
]

readme_lines = [
    'NASA Earthdata MODIS HDF Products\n',
    'Catalog: https://search.earthdata.nasa.gov/search?fst0=Cryosphere\n\n',
]

for product in products:
    short_name = product['short_name']
    start = product['start']
    end = product['end']
    sub = modis_dir / product['label']
    sub.mkdir(exist_ok=True)

    print(f"Searching {short_name} ({start} to {end})...")
    print(f"  Browser check: {product['search_url']}")
    try:
        kwargs = {
            'short_name': short_name,
            'temporal': (start, end),
            'bounding_box': (-180, 60, 180, 90),
        }
        
        results = earthaccess.search_data(**kwargs)

        if not results and short_name == 'MOD11A3':
            # MOD11A3 is land-only; broaden the box if Arctic-ocean-only filtering misses land tiles.
            print('  No Arctic-box granules found; retrying with a broader northern land box...')
            kwargs['bounding_box'] = (-180, 45, 180, 90)
            results = earthaccess.search_data(**kwargs)

        if results:
            n = min(2, len(results))
            earthaccess.download(results[:n], str(sub))
            print(f'  Downloaded {n} granule(s) -> {sub}')
        else:
            print(f'  No granules found for {short_name}; use the browser link above to inspect availability.')
    except Exception as e:
        print(f'  Error: {e}')
        print(f"  Try manually: {product['search_url']}")

    readme_lines.append(
        f"{short_name}: {product['search_url']}\n"
        f"  Date window: {start} to {end}\n"
        f"  Local folder: {product['label']}\n\n"
    )

(modis_dir / 'README.txt').write_text(''.join(readme_lines))
print(f'\nDone -> {modis_dir}')

Searching MOD29 (2023-09-16 to 2023-09-16)...
  Browser check: https://search.earthdata.nasa.gov/search?q=MOD29


QUEUEING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2 [00:00<?, ?it/s]

  Downloaded 2 granule(s) -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_03_04_06__MODIS_HDF_Products/VIZ01_02_05__MOD29_SeaIce
Searching MOD10A1 (2023-09-16 to 2023-09-16)...
  Browser check: https://search.earthdata.nasa.gov/search?q=MOD10A1


...

  Downloaded 2 granule(s) -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_03_04_06__MODIS_HDF_Products/VIZ04__MOD11A3_LST_Monthly
Searching MCD43A3 (2023-09-16 to 2023-09-16)...
  Browser check: https://search.earthdata.nasa.gov/search?q=MCD43A3


QUEUEING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2 [00:00<?, ?it/s]

  Downloaded 2 granule(s) -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_03_04_06__MODIS_HDF_Products/VIZ06__MCD43A3_Albedo

Done -> /Users/joeysandoval/Desktop/idea5_cryosphere_data/VIZ01_03_04_06__MODIS_HDF_Products


---
## Summary

In [9]:
print(f'\n{"="*58}')
print('  Idea 5 Cryosphere Data — Download Summary')
print(f'{"="*58}')
print(f'  Location: {SAVE_DIR}\n')

total = 0
for folder in sorted(SAVE_DIR.iterdir()):
    if folder.is_dir():
        files = [f for f in folder.rglob('*') if f.is_file()]
        sz = sum(f.stat().st_size for f in files)
        total += sz
        sz_str = f'{sz/1024:.1f} KB' if sz < 1e6 else f'{sz/1e6:.1f} MB'
        print(f'  {folder.name}/  ({len(files)} files, {sz_str})')
        for f in sorted(files):
            print(f'    {f.name}')
        print()

t_str = f'{total/1024:.1f} KB' if total < 1e6 else f'{total/1e6:.1f} MB'
print(f'  Total downloaded: {t_str}')
print(f'{"="*58}')


  Idea 5 Cryosphere Data — Download Summary
  Location: /Users/joeysandoval/Desktop/idea5_cryosphere_data

  VIZ01_02_05__NSIDC_SeaIceIndex_v4/  (27 files, 3.7 MB)
    N_01_extent_v4.0.csv
    N_02_extent_v4.0.csv
    N_03_extent_v4.0.csv
    N_04_extent_v4.0.csv
    N_05_extent_v4.0.csv
    N_06_extent_v4.0.csv
    N_07_extent_v4.0.csv
    N_08_extent_v4.0.csv
    N_09_extent_v4.0.csv
    N_10_extent_v4.0.csv
    N_11_extent_v4.0.csv
    N_12_extent_v4.0.csv
    N_seaice_extent_daily_v4.0.csv
    README.txt
    S_01_extent_v4.0.csv
    S_02_extent_v4.0.csv
    S_03_extent_v4.0.csv
    S_04_extent_v4.0.csv
    S_05_extent_v4.0.csv
    S_06_extent_v4.0.csv
    S_07_extent_v4.0.csv
    S_08_extent_v4.0.csv
    S_09_extent_v4.0.csv
    S_10_extent_v4.0.csv
    S_11_extent_v4.0.csv
    S_12_extent_v4.0.csv
    S_seaice_extent_daily_v4.0.csv

  VIZ01_03_04_06__MODIS_HDF_Products/  (9 files, 103.6 MB)
    README.txt
    MOD29.A2023259.0105.061.2023259131937.hdf
    MOD29.A2023259.0110.061.2